In [1]:
import cv2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

segment_dir = 'output_segments'

segment_files = glob.glob(os.path.join(segment_dir, '*.png'))
len(segment_files)

23

In [25]:
from skimage.feature import graycomatrix, graycoprops
from skimage import io, color
from skimage.util import img_as_ubyte

glcm_features = [
    "contrast",
    "dissimilarity",
    "homogeneity",
    "energy",
    "correlation",
    "ASM"
]

def get_glcm(image):

    # Convert to grayscale if needed
    if image.ndim == 3:
        image = color.rgb2gray(image)

    # Convert to 8-bit unsigned integers (0–255)
    image = img_as_ubyte(image)

    # Compute GLCM
    glcm = graycomatrix(
        image,
        distances=[5],                  # pixel distance
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )

    print("GLCM shape:", glcm.shape)
    
    features = {graycoprops(glcm, feature).mean() for feature in glcm_features}
    
    for k, v in features.items():
        features[k] = np.mean(v)
    
    print(features)
    feature_vector = [np.mean(value) for value in features.values()]
    print("Feature vector:", feature_vector)
    return feature_vector, features

In [9]:
from skimage.feature import local_binary_pattern

def multiscale_lbp(image):
    lbp_scales = [
        (3, 8),     # lath edges,
        # (2, 16),    # sub-lath texture
        # (3, 24),    # lath bundles
        # (4, 32),    # packet/block structure
        # (5, 40)     # coarse bainitic regions (optional)
    ]

    multi_lbp_hist = []

    for radius, n_points in lbp_scales:
        lbp = local_binary_pattern(
            image,
            n_points,
            radius,
            method='uniform'
        )

        n_bins = n_points + 2  # uniform LBP

        lbp_hist, _ = np.histogram(
            lbp.ravel(),
            bins=n_bins,
            range=(0, n_bins),
            density=True
        )

        lbp_hist = lbp_hist.astype("float")
        lbp_hist /= (lbp_hist.sum() + 1e-6)

        multi_lbp_hist.append(lbp_hist)

    # 🔑 concatenate all scales into ONE vector
    return np.hstack(multi_lbp_hist)

In [10]:
import os
import glob
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

def get_all_image_statistics(segment_dir = 'output_segments'):
    segment_files = glob.glob(os.path.join(segment_dir, '*.png'))
    statistics = {
        'martensite': 
            {"lbp" : [],
            "img_hist" : [],
            "img" : [],
            "multiscale_lbp" : [],
            "glcm_features" : [],
            "glcm_full" : []
            },
        'bainite':
            {"lbp" : [],
            "img_hist" : [],
            "img" : [],
            "multiscale_lbp" : [],
            "glcm_features" : [],
            "glcm_full" : []
            },
    }

    for filename in segment_files:
        parts = filename.split("_")
        label = parts[-2].lower()  # second last part is label
        print(label)

        if label not in statistics:
            print(f"Skipping unknown label '{label}' in file {filename}")
            continue

        # --- Load image ---
        image = cv2.imread(filename)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
        # --- Get img hist ---
        img_hist, _ = np.histogram(
            gray.ravel(),
            bins=256,
            range=(1, 256),
            density=True
        )
        
        # Store histogram
        if label in statistics:
            statistics[label]["img_hist"].append(img_hist)
        else:
            print(f"Unknown label: {label}")

        multiscale_lbp_hist = multiscale_lbp(gray)
        statistics[label]["multiscale_lbp"].append(multiscale_lbp_hist)
        glcm_avg_feats, feats = get_glcm(gray)
        statistics[label]["glcm_features"].append(glcm_avg_feats)
        statistics[label]["glcm_full"].append(feats)
        statistics[label]["img"].append(gray)

        # # --- Optional: visualize ---
        # plt.figure(figsize=(10,4))
        # plt.subplot(1,2,1)
        # plt.title('Original (gray)')
        # plt.imshow(gray, cmap='gray')
        # plt.axis('off')

        # plt.subplot(1,2,2)
        # plt.title('LBP')
        # plt.imshow(lbp, cmap='gray')
        # plt.axis('off')

        # plt.tight_layout()
        # plt.show()
    return statistics

In [11]:
statistics = get_all_image_statistics()

bainite
GLCM shape: (256, 256, 1, 4)
{'contrast': np.float64(1726.4659982574449), 'dissimilarity': np.float64(23.15328218538591), 'homogeneity': np.float64(0.37078895984986815), 'energy': np.float64(0.3456595269625046), 'correlation': np.float64(0.8648558697376746), 'ASM': np.float64(0.11952492816804007)}
Feature vector: [np.float64(1726.4659982574449), np.float64(23.15328218538591), np.float64(0.37078895984986815), np.float64(0.3456595269625046), np.float64(0.8648558697376746), np.float64(0.11952492816804007)]
bainite
GLCM shape: (256, 256, 1, 4)
{'contrast': np.float64(1708.5341523584193), 'dissimilarity': np.float64(21.830327417570963), 'homogeneity': np.float64(0.434302730352936), 'energy': np.float64(0.4100136002124872), 'correlation': np.float64(0.8799952896973418), 'ASM': np.float64(0.16814469206059135)}
Feature vector: [np.float64(1708.5341523584193), np.float64(21.830327417570963), np.float64(0.434302730352936), np.float64(0.4100136002124872), np.float64(0.8799952896973418), n

In [13]:
from scipy.stats import entropy, skew

def hist_stats(hist):
    hist = hist / (hist.sum() + 1e-8)
    bins = np.arange(len(hist))

    mean = np.sum(bins * hist)
    var = np.sum((bins - mean) ** 2 * hist)
    std = np.sqrt(var)

    return [
        mean,
        std,
        entropy(hist),
        np.sum(hist ** 2),  # energy
        skew(hist)
    ]

def glcm_stats(glcm_vec):
    return glcm_vec

In [14]:
X_small = []
y = []

for label, class_data in statistics.items():
    for img_hist, lbp_vec, glcm_vec in zip(
        class_data['img_hist'],
        class_data['multiscale_lbp'],
        class_data['glcm_features']
    ):
        features = np.hstack([
            hist_stats(img_hist),                                   # intensity
            hist_stats(
                lbp_vec
            ),                                                       # texture
            glcm_stats(glcm_vec)                                     # structure
        ])

        X_small.append(features)
        y.append(label)

X_small = np.vstack(X_small)
y = np.array(y)


In [15]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=0.05,
        max_iter=500,
        class_weight='balanced'
    ))
])

loo = LeaveOneOut()
y_pred = cross_val_predict(pipe, X_small, y, cv=loo)

print(classification_report(y, y_pred))

              precision    recall  f1-score   support

     bainite       0.80      0.89      0.84         9
  martensite       0.92      0.86      0.89        14

    accuracy                           0.87        23
   macro avg       0.86      0.87      0.87        23
weighted avg       0.87      0.87      0.87        23



In [16]:
from sklearn.metrics import confusion_matrix, balanced_accuracy_score

print("Balanced accuracy:", balanced_accuracy_score(y, y_pred))
print(confusion_matrix(y, y_pred))

Balanced accuracy: 0.873015873015873
[[ 8  1]
 [ 2 12]]


In [17]:
coefs = []

for train_idx, _ in loo.split(X_small):
    pipe.fit(X_small[train_idx], y[train_idx])
    coefs.append(pipe.named_steps['clf'].coef_[0])

coefs = np.vstack(coefs)  # shape: (n_samples, n_features)

# ---------------------------
# 3️⃣ Compute stability metrics
# ---------------------------
coef_mean = coefs.mean(axis=0)
coef_std = coefs.std(axis=0)
sign_consistency = np.mean(np.sign(coefs) == np.sign(coef_mean), axis=0)
stability_ratio = np.abs(coef_mean) / (coef_std + 1e-6)

In [27]:
feature_names = []

# 1️⃣ Intensity features
n_intensity = 5
intensity_names = ['intensity_mean', 'intensity_std', 'intensity_entropy', 'intensity_energy', 'intensity_skew']
feature_names.extend(intensity_names)

# 2️⃣ LBP features
lbp_stat_names = ['lbp_mean', 'lbp_std', 'lbp_entropy', 'lbp_energy', 'lbp_skew']

feature_names.extend(lbp_stat_names)

# 3️⃣ GLCM features
glcm_vec_example = statistics[list(statistics.keys())[0]]['glcm_features'][0]
glcm_names = [f'glcm_{i}' for i in glcm_features]
feature_names.extend(glcm_names)

In [28]:
import pandas as pd

df_features = pd.DataFrame({
    'feature': feature_names,
    'coef_mean': coef_mean,
    'coef_std': coef_std,
    'sign_consistency': sign_consistency,
    'stability_ratio': stability_ratio
})


In [29]:
# Sort by stability_ratio for importance
df_features = df_features.sort_values('stability_ratio', ascending=False)

# ---------------------------
# 5️⃣ Output
# ---------------------------
print("Top features by stability ratio:")
print(df_features.head(20))
# Optional: group features
def feature_group(name):
    if name.startswith('intensity'):
        return 'Intensity'
    elif name.startswith('lbp'):
        return 'LBP'
    elif name.startswith('glcm'):
        return 'GLCM'
    else:
        return 'Other'
# Summary by group
df_features['group'] = df_features['feature'].apply(feature_group)
group_summary = df_features.groupby('group')['stability_ratio'].mean()
print("\nMean stability ratio by feature group:")
print(group_summary)


Top features by stability ratio:
               feature  coef_mean  coef_std  sign_consistency  stability_ratio
2    intensity_entropy  -0.220294  0.003925          1.000000        56.106819
1        intensity_std  -0.286534  0.006627          1.000000        43.231522
4       intensity_skew   0.199441  0.005835          1.000000        34.172207
3     intensity_energy   0.187239  0.007731          1.000000        24.217244
8           lbp_energy   0.036594  0.006461          1.000000         5.662845
9             lbp_skew  -0.059742  0.011008          1.000000         5.426427
11  glcm_dissimilarity  -0.034249  0.006767          1.000000         5.060259
7          lbp_entropy  -0.032175  0.006700          1.000000         4.801794
15            glcm_ASM   0.025995  0.006226          1.000000         4.174941
14    glcm_correlation  -0.023573  0.007355          1.000000         3.204616
5             lbp_mean   0.022906  0.007419          1.000000         3.086980
0       intensity_m